# Notebook 13 – Saving and Loading Models

**Flow covered in this notebook**

```
Hugging Face Hub
       ↓
Download
       ↓
Local Directory
       ↓
Load
       ↓
Inference
```

Install once if needed: `pip install transformers torch huggingface_hub`

## Topic 1: Downloading a Model

**Concept**
`from_pretrained("model-name")` downloads the model weights, config, and tokenizer files from the Hugging Face Hub the first time you call it.

**Why it's needed**
Every pre-trained model starts as files on the Hub. Downloading is the first step before you can save, move, or run the model anywhere else.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

print("Model class :", type(model).__name__)
print("Parameters  :", f"{model.num_parameters():,}")
print("Labels      :", model.config.id2label)

**Expected output**
```
Model class : DistilBertForSequenceClassification
Parameters  : 66,955,010
Labels      : {0: 'NEGATIVE', 1: 'POSITIVE'}
```

**What this example tells us**
One line of code pulled a ~67M-parameter model from the Hub and gave us a ready-to-use model and tokenizer.

**AI/ML Example**
Downloading a sentiment classifier that was already fine-tuned on SST-2, instead of training one from scratch.

**Business Example**
A support team downloads a ready-made sentiment model to score customer feedback without building a model first.

**AI/ML Engineer Use Case**
Download once during development, then decide where the files should live (cache, local folder, or artifact store) for the rest of the project.

## Topic 2: Local Model Cache

**Concept**
After the first download, Hugging Face stores the files in a local cache folder (default: `~/.cache/huggingface/hub`). The next `from_pretrained()` call reuses the cache instead of downloading again.

**Why it's needed**
The cache saves time and bandwidth, and it is the reason a model can load again without internet once it has been downloaded.

In [ ]:
from huggingface_hub import scan_cache_dir

cache_info = scan_cache_dir()

for repo in cache_info.repos:
    if repo.repo_id == MODEL_NAME:
        print("Repo     :", repo.repo_id)
        print("Size (MB):", round(repo.size_on_disk / 1024 / 1024, 1))
        print("Location :", repo.repo_path)

**Expected output** (size and path differ on your machine)
```
Repo     : distilbert-base-uncased-finetuned-sst-2-english
Size (MB): ~255
Location : /home/<user>/.cache/huggingface/hub/models--distilbert-base-uncased-finetuned-sst-2-english
```

**What this example tells us**
The model files already sit on disk after the first download, and we can inspect how much space they use.

**AI/ML Example**
Re-running a notebook loads the model in seconds because the files come from cache, not the network.

**Business Example**
A team of 10 developers on a shared build server avoids downloading the same model 10 times.

**AI/ML Engineer Use Case**
Check cache size on CI servers and containers, and clean it (`huggingface-cli delete-cache`) when disk space gets tight.

## Topic 3: Saving Models

**Concept**
`model.save_pretrained(folder)` writes the model weights (`model.safetensors`) and its configuration (`config.json`) into a folder you choose.

**Why it's needed**
The cache is managed by Hugging Face and is hard to control. A saved folder is a clean, portable copy you can version, copy, or ship with your application.

In [ ]:
import os

SAVE_DIR = "./saved_models/sst2_distilbert"

model.save_pretrained(SAVE_DIR)

print("Saved to:", SAVE_DIR)
print("Files   :", sorted(os.listdir(SAVE_DIR)))

**Expected output**
```
Saved to: ./saved_models/sst2_distilbert
Files   : ['config.json', 'model.safetensors']
```

**What this example tells us**
The model is now a plain folder with two files: the architecture settings and the trained weights.

**AI/ML Example**
After fine-tuning your own model, `save_pretrained()` is how you keep the result.

**Business Example**
A company stores an approved model folder in its internal file storage so everyone uses the same version.

**AI/ML Engineer Use Case**
Save each trained or approved model into a versioned folder (for example `models/sentiment/v1`) so deployments are repeatable.

## Topic 4: Saving Tokenizers

**Concept**
`tokenizer.save_pretrained(folder)` saves the vocabulary and tokenizer settings. Save it in the same folder as the model.

**Why it's needed**
A model only understands the token IDs it was trained with. If the tokenizer is missing or different, the same text turns into different IDs and the predictions become wrong.

In [ ]:
tokenizer.save_pretrained(SAVE_DIR)

print("Files now in folder:")
for name in sorted(os.listdir(SAVE_DIR)):
    print(" -", name)

**Expected output** (exact list can vary slightly by library version)
```
Files now in folder:
 - config.json
 - model.safetensors
 - special_tokens_map.json
 - tokenizer_config.json
 - vocab.txt
```

**What this example tells us**
The folder now holds everything needed to run the model: weights, config, and tokenizer files.

**AI/ML Example**
Fine-tuning with a custom tokenizer (extra special tokens, for example) and saving both together so they never get out of sync.

**Business Example**
Shipping a model without its tokenizer is like shipping software without its config file. It will not behave as tested.

**AI/ML Engineer Use Case**
Always save model and tokenizer in the same folder and treat that folder as one deployable unit.

## Topic 5: Loading Models from Local Directories

**Concept**
Pass the folder path to `from_pretrained()` instead of a Hub name. Hugging Face reads the files from disk.

**Why it's needed**
This is how saved models are reused in later sessions, other machines, containers, and production services.

In [ ]:
local_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
local_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)

print("Loaded from        :", SAVE_DIR)
print("Same parameter count:", local_model.num_parameters() == model.num_parameters())
print("Same labels         :", local_model.config.id2label == model.config.id2label)

**Expected output**
```
Loaded from         : ./saved_models/sst2_distilbert
Same parameter count: True
Same labels         : True
```

**What this example tells us**
The model loaded from the local folder is identical to the one downloaded from the Hub.

**AI/ML Example**
Loading a fine-tuned checkpoint from `./outputs/best_model` to run evaluation.

**Business Example**
An approved model folder is copied to every production server and loaded the same way everywhere.

**AI/ML Engineer Use Case**
Use a config value like `MODEL_PATH` so the same code loads either a Hub name (development) or a local folder (production).

## Topic 6: Offline Inference Concept

**Concept**
Offline inference means running the model with no internet connection. Use a local folder plus `local_files_only=True` so Hugging Face never tries to reach the Hub.

**Why it's needed**
Production servers, secure networks, and edge devices often have no internet access. The model must work using only local files.

In [ ]:
import torch

offline_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR, local_files_only=True)
offline_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR, local_files_only=True)
offline_model.eval()

text = "The service was fast and the support team was helpful."
inputs = offline_tokenizer(text, return_tensors="pt")

with torch.no_grad():
    logits = offline_model(**inputs).logits

probs = torch.softmax(logits, dim=-1)[0]
pred_id = int(probs.argmax())

print("Text      :", text)
print("Prediction:", offline_model.config.id2label[pred_id])
print("Confidence:", round(float(probs[pred_id]), 4))

**Expected output** (confidence is approximate)
```
Text      : The service was fast and the support team was helpful.
Prediction: POSITIVE
Confidence: 0.9998
```

**What this example tells us**
The model gave a prediction using only local files. No Hub connection was needed. To test it for real, disconnect your internet and re-run this cell.

**AI/ML Example**
Running inference inside a Docker container that has no outbound network access.

**Business Example**
A bank runs a text classifier on an isolated server where internet access is blocked by policy.

**AI/ML Engineer Use Case**
Set `local_files_only=True` (or the `HF_HUB_OFFLINE=1` environment variable) in production so a missing file fails loudly instead of silently trying to download.

## Topic 7: Full Flow and Why Local Storage Matters in Enterprise

**Concept**
Putting it together: download from the Hub once, save to a local directory, then load and run inference from that directory.

```
Hugging Face Hub  →  Download  →  Local Directory  →  Load  →  Inference
```

**Why it's needed (enterprise view)**
- **Security:** production systems often cannot reach the public internet.
- **Control:** only reviewed, approved models are used.
- **Reproducibility:** a Hub model can change or be removed; a saved copy never does.
- **Reliability:** no dependency on Hub uptime or network speed.
- **Compliance:** data and model stay inside the company network.
- **Speed and cost:** no repeated downloads across servers.

In [ ]:
def run_local_pipeline(text, model_dir=SAVE_DIR):
    print("1. Source      : Hugging Face Hub ->", MODEL_NAME)
    print("2. Downloaded  : done (Topic 1)")
    print("3. Local folder:", model_dir)

    tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_dir, local_files_only=True)
    mdl.eval()
    print("4. Loaded      : from local directory")

    inputs = tok(text, return_tensors="pt")
    with torch.no_grad():
        probs = torch.softmax(mdl(**inputs).logits, dim=-1)[0]
    pred_id = int(probs.argmax())
    print("5. Inference   :", mdl.config.id2label[pred_id], f"({float(probs[pred_id]):.4f})")

run_local_pipeline("The delivery was late and nobody answered my emails.")

**Expected output** (confidence is approximate)
```
1. Source      : Hugging Face Hub -> distilbert-base-uncased-finetuned-sst-2-english
2. Downloaded  : done (Topic 1)
3. Local folder: ./saved_models/sst2_distilbert
4. Loaded      : from local directory
5. Inference   : NEGATIVE (0.9995)
```

**What this example tells us**
Once the model is saved, every later step uses only the local folder. The Hub is needed just once.

**AI/ML Example**
One script downloads and saves the model on a build machine; the inference service only ever loads from the saved folder.

**Business Example**
IT approves a model once, stores it in an internal repository, and every department uses that exact version.

**AI/ML Engineer Use Case**
Build a "download and save" step into your CI/CD pipeline, package the saved folder into the Docker image, and run the service with offline mode on.